In [3]:
import sys
print(sys.executable)

/opt/anaconda3/envs/voting_power/bin/python


In [4]:
from math import comb
import pandas as pd


def exact_power_indices(weights, quota, names=None):
    """
    Exact Banzhaf and Shapley-Shubik indices for an integer-weighted voting game.

    weights: list of voting weights
    quota: minimum total weight needed to pass
    names: optional list of player names
    """

    weights = [int(w) for w in weights]
    quota = int(quota)
    n = len(weights)

    if names is None:
        names = [f"Player {i+1}" for i in range(n)]

    if len(names) != n:
        raise ValueError("The number of names must equal the number of weights.")

    if quota <= 0 or quota > sum(weights):
        raise ValueError("Quota must be between 1 and the total voting weight.")

    banzhaf_swings = []
    shapley_shubik = []

    for i, wi in enumerate(weights):

        others = weights[:i] + weights[i+1:]

        # ---------------------------------------------------------
        # Exact Banzhaf calculation
        #
        # dp_weight[s] = number of coalitions of the other players
        # whose total weight is exactly s.
        #
        # Only losing coalitions (s < quota) need to be stored.
        # ---------------------------------------------------------

        dp_weight = [0] * quota
        dp_weight[0] = 1

        for w in others:
            for s in range(quota - 1 - w, -1, -1):
                dp_weight[s + w] += dp_weight[s]

        lower = max(0, quota - wi)

        # Player i is critical when:
        # quota - wi <= coalition weight < quota
        swings_i = sum(dp_weight[lower:quota])
        banzhaf_swings.append(swings_i)

        # ---------------------------------------------------------
        # Exact Shapley-Shubik calculation
        #
        # dp_size_weight[k][s] = number of coalitions containing
        # exactly k other players and having total weight s.
        # ---------------------------------------------------------

        dp_size_weight = [[0] * quota for _ in range(n)]
        dp_size_weight[0][0] = 1

        processed = 0

        for w in others:
            for k in range(processed, -1, -1):
                for s in range(quota - 1 - w, -1, -1):
                    dp_size_weight[k + 1][s + w] += dp_size_weight[k][s]
            processed += 1

        ssi_i = 0.0

        for k in range(n):
            pivotal_coalitions = sum(
                dp_size_weight[k][s]
                for s in range(lower, quota)
            )

            # Each size-k predecessor coalition has probability
            # 1 / [n * C(n-1,k)] in a random ordering.
            if pivotal_coalitions:
                ssi_i += pivotal_coalitions / (n * comb(n - 1, k))

        shapley_shubik.append(ssi_i)

    total_swings = sum(banzhaf_swings)

    normalized_banzhaf = [
        swings / total_swings for swings in banzhaf_swings
    ]

    absolute_banzhaf = [
        swings / (2 ** (n - 1)) for swings in banzhaf_swings
    ]

    results = pd.DataFrame({
        "player": names,
        "weight": weights,
        "Banzhaf_swings": banzhaf_swings,
        "Banzhaf_absolute": absolute_banzhaf,
        "Banzhaf_normalized": normalized_banzhaf,
        "Shapley_Shubik": shapley_shubik
    })

    results["Banzhaf_normalized_percent"] = (
        100 * results["Banzhaf_normalized"]
    )

    results["Shapley_Shubik_percent"] = (
        100 * results["Shapley_Shubik"]
    )

    return results

In [13]:
# Standard UNSC with a permanent members abstaining in advance

abstention_records = []

for a in range(6):
    active_permanent = 5 - a
    permanent_weight = 7 - a

    weights = (
        [permanent_weight] * active_permanent
        + [1] * 10
    )

    # Nine affirmative votes are still required.
    quota = (
        active_permanent * permanent_weight
        + (4 + a)
    )

    calculation = exact_power_indices(
        weights=weights,
        quota=quota
    )

    if active_permanent > 0:
        permanent_result = calculation.iloc[0]

        abstention_records.append({
            "P5_abstentions": a,
            "member_type": "Active permanent",
            "multiplicity": active_permanent,
            "weight": permanent_weight,
            "quota": quota,
            "Banzhaf_percent":
                permanent_result["Banzhaf_normalized_percent"],
            "Shapley_Shubik_percent":
                permanent_result["Shapley_Shubik_percent"]
        })

    nonpermanent_result = calculation.iloc[active_permanent]

    abstention_records.append({
        "P5_abstentions": a,
        "member_type": "Non-permanent",
        "multiplicity": 10,
        "weight": 1,
        "quota": quota,
        "Banzhaf_percent":
            nonpermanent_result["Banzhaf_normalized_percent"],
        "Shapley_Shubik_percent":
            nonpermanent_result["Shapley_Shubik_percent"]
    })

results_abstentions = pd.DataFrame(abstention_records)

results_abstentions.round(6)

,P5_abstentions,member_type,multiplicity,weight,quota,Banzhaf_percent,Shapley_Shubik_percent
0,0,Active permanent,5,7,39,16.692913,19.627040
1,0,Non-permanent,10,1,39,1.653543,0.186480
2,1,Active permanent,4,6,29,16.736621,23.251748
3,1,Non-permanent,10,1,29,3.305352,0.699301
4,2,Active permanent,3,5,21,15.963606,26.806527
5,2,Non-permanent,10,1,21,5.210918,1.958042
6,3,Active permanent,2,4,15,14.765101,28.787879
7,3,Non-permanent,10,1,15,7.046980,4.242424
8,4,Active permanent,1,3,11,13.461538,27.272727
9,4,Non-permanent,10,1,11,8.653846,7.272727


In [14]:
# Standard UNSC: regional-bloc voting system
# Order: Africa, Asia-Pacific, GRULAC, WEOG, EEG

weights = [3, 9, 2, 23, 8]
quota = 39

results_standard_regions = exact_power_indices(
    weights=weights,
    quota=quota
)

results_standard_regions[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(6)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,3,0.000000,0.000000
1,Player 2,9,33.333333,33.333333
2,Player 3,2,0.000000,0.000000
3,Player 4,23,33.333333,33.333333
4,Player 5,8,33.333333,33.333333


In [15]:
# Kofi Annan Models A and B
# Players 1-5: members with veto
# Players 6-24: members without veto

weights = [10] * 5 + [1] * 19
quota = 60

results_annan = exact_power_indices(
    weights=weights,
    quota=quota
)

results_annan[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(6)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,10,11.731663,19.057971
1,Player 2,10,11.731663,19.057971
2,Player 3,10,11.731663,19.057971
3,Player 4,10,11.731663,19.057971
4,Player 5,10,11.731663,19.057971
5,Player 6,1,2.175878,0.247902
6,Player 7,1,2.175878,0.247902
7,Player 8,1,2.175878,0.247902
8,Player 9,1,2.175878,0.247902
9,Player 10,1,2.175878,0.247902


In [16]:
# Kofi Annan proposals: regional-bloc voting system
# Order: Africa, Asia-Pacific, GRULAC, WEOG, EEG

weights = [6, 15, 3, 34, 11]
quota = 60

results_annan_regions = exact_power_indices(
    weights=weights,
    quota=quota
)

results_annan_regions[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(6)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,6,0.000000,0.000000
1,Player 2,15,33.333333,33.333333
2,Player 3,3,0.000000,0.000000
3,Player 4,34,33.333333,33.333333
4,Player 5,11,33.333333,33.333333


In [17]:
# African Union proposal
# Players 1-11: permanent members with veto
# Players 12-26: non-permanent members

weights = [11] * 11 + [1] * 15
quota = 126

results_au = exact_power_indices(
    weights=weights,
    quota=quota
)

results_au[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(9)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,11,8.705438,9.089303
1,Player 2,11,8.705438,9.089303
2,Player 3,11,8.705438,9.089303
3,Player 4,11,8.705438,9.089303
4,Player 5,11,8.705438,9.089303
5,Player 6,11,8.705438,9.089303
6,Player 7,11,8.705438,9.089303
7,Player 8,11,8.705438,9.089303
8,Player 9,11,8.705438,9.089303
9,Player 10,11,8.705438,9.089303


In [18]:
# African Union proposal: regional-bloc voting system
# Order: Africa, Asia-Pacific, GRULAC, WEOG, EEG

weights = [27, 36, 14, 46, 13]
quota = 126

results_au_regions = exact_power_indices(
    weights=weights,
    quota=quota
)

results_au_regions[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(6)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,27,20.0,20.0
1,Player 2,36,20.0,20.0
2,Player 3,14,20.0,20.0
3,Player 4,46,20.0,20.0
4,Player 5,13,20.0,20.0


In [19]:
# G4 and UfC proposals
# Players 1-5: members with veto
# Players 6-25: members without veto

weights = [12] * 5 + [1] * 20
quota = 70

results_g4_ufc = exact_power_indices(
    weights=weights,
    quota=quota
)

results_g4_ufc[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(6)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,12,12.506180,19.246377
1,Player 2,12,12.506180,19.246377
2,Player 3,12,12.506180,19.246377
3,Player 4,12,12.506180,19.246377
4,Player 5,12,12.506180,19.246377
5,Player 6,1,1.873455,0.188406
6,Player 7,1,1.873455,0.188406
7,Player 8,1,1.873455,0.188406
8,Player 9,1,1.873455,0.188406
9,Player 10,1,1.873455,0.188406


In [20]:
# G4 and UfC proposals: regional-bloc voting system
# Order: Africa, Asia-Pacific, GRULAC, WEOG, EEG

weights = [6, 17, 4, 39, 14]
quota = 70

results_g4_ufc_regions = exact_power_indices(
    weights=weights,
    quota=quota
)

results_g4_ufc_regions[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(6)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,6,0.000000,0.000000
1,Player 2,17,33.333333,33.333333
2,Player 3,4,0.000000,0.000000
3,Player 4,39,33.333333,33.333333
4,Player 5,14,33.333333,33.333333


In [21]:
# L.69/CARICOM proposal
# Players 1-11: permanent members with veto
# Players 12-27: non-permanent members
#
# At least 17 of 27 members must vote affirmatively.
# Therefore the corrected weighted-game quota is 138.

weights = [12] * 11 + [1] * 16
quota = 138

results_l69 = exact_power_indices(
    weights=weights,
    quota=quota
)

results_l69[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(9)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,12,8.460796,9.087863
1,Player 2,12,8.460796,9.087863
2,Player 3,12,8.460796,9.087863
3,Player 4,12,8.460796,9.087863
4,Player 5,12,8.460796,9.087863
5,Player 6,12,8.460796,9.087863
6,Player 7,12,8.460796,9.087863
7,Player 8,12,8.460796,9.087863
8,Player 9,12,8.460796,9.087863
9,Player 10,12,8.460796,9.087863


In [22]:
# L.69/CARICOM proposal: regional-bloc voting system
# Order: Africa, Asia-Pacific, GRULAC, WEOG, EEG, SIDS

weights = [29, 39, 15, 50, 14, 1]
quota = 138

results_l69_regions = exact_power_indices(
    weights=weights,
    quota=quota
)

results_l69_regions[
    [
        "player",
        "weight",
        "Banzhaf_normalized_percent",
        "Shapley_Shubik_percent"
    ]
].round(6)

,player,weight,Banzhaf_normalized_percent,Shapley_Shubik_percent
0,Player 1,29,20.0,20.0
1,Player 2,39,20.0,20.0
2,Player 3,15,20.0,20.0
3,Player 4,50,20.0,20.0
4,Player 5,14,20.0,20.0
5,Player 6,1,0.0,0.0


In [23]:
calculations = {
    "Standard regions": results_standard_regions,
    "Annan members": results_annan,
    "Annan regions": results_annan_regions,
    "AU members": results_au,
    "AU regions": results_au_regions,
    "G4/UfC members": results_g4_ufc,
    "G4/UfC regions": results_g4_ufc_regions,
    "L.69/CARICOM members": results_l69,
    "L.69/CARICOM regions": results_l69_regions
}

validation = pd.DataFrame([
    {
        "calculation": label,
        "Banzhaf_total":
            result["Banzhaf_normalized_percent"].sum(),
        "Shapley_Shubik_total":
            result["Shapley_Shubik_percent"].sum()
    }
    for label, result in calculations.items()
])

validation.round(10)

,calculation,Banzhaf_total,Shapley_Shubik_total
0,Standard regions,100.0,100.0
1,Annan members,100.0,100.0
2,Annan regions,100.0,100.0
3,AU members,100.0,100.0
4,AU regions,100.0,100.0
5,G4/UfC members,100.0,100.0
6,G4/UfC regions,100.0,100.0
7,L.69/CARICOM members,100.0,100.0
8,L.69/CARICOM regions,100.0,100.0
